## 1. Qual é a receita total (em R$) somada de todos os filmes da base?

In [0]:
display(spark.sql("""
    SELECT SUM(receita_brl) AS receita_total_brl
    FROM gold.fact_movies_performance
"""))

## 2. Quais são os 5 filmes com maior popularidade?

In [0]:
display(spark.sql("""
    SELECT m.titulo, f.popularidade
    FROM gold.dim_movies m
    JOIN gold.fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
    ORDER BY f.popularidade DESC
    LIMIT 5
"""))

# Devido ao column shift nos arquivos CSV brutos, alguns anos de lançamento (ex: 2020, 2018) vazaram para a métrica de popularidade. 
# Optei por não aplicar um filtro excludente rigoroso para números inteiros na casa dos milhares, pois isso geraria falsos positivos e eliminaria filmes com popularidade real alta (como Blue Beetle, com ~2994), priorizando a retenção de dados válidos. 

## 3. Quantos filmes cada gênero possui? Liste do maior para o menor.

In [0]:
display(spark.sql("""
    SELECT g.nome_genero, COUNT(b.sk_movie_id) AS qtd_filmes
    FROM gold.dim_genres g
    JOIN gold.bridge_movie_genre b ON g.sk_genre_id = b.sk_genre_id
    GROUP BY g.nome_genero
    ORDER BY qtd_filmes DESC
"""))

## 4. 10 filmes de maior receita com a posição no ranking (RANK()).

In [0]:
display(spark.sql("""
    SELECT 
        m.titulo, 
        f.receita_usd, 
        f.receita_brl,
        RANK() OVER (ORDER BY f.receita_usd DESC) AS ranking
    FROM gold.dim_movies m
    JOIN gold.fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
    WHERE f.receita_usd IS NOT NULL
    ORDER BY ranking
    LIMIT 10
"""))

## 5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos?

In [0]:
display(spark.sql("""
    WITH max_ano AS (
        SELECT MAX(ano_lancamento) as ano_max
        FROM gold.dim_movies
        WHERE status_filme = 'Lançado' AND data_lancamento <= current_date()
    )
    SELECT p.nome_pessoa, COUNT(b.sk_movie_id) AS qtd_participacoes
    FROM gold.dim_people p
    JOIN gold.bridge_movie_person b ON p.sk_person_id = b.sk_person_id
    JOIN gold.dim_movies m ON b.sk_movie_id = m.sk_movie_id
    CROSS JOIN max_ano
    WHERE p.tipo_pessoa = 'Ator' 
      AND m.ano_lancamento >= (max_ano.ano_max - 2)
    GROUP BY p.nome_pessoa
    ORDER BY qtd_participacoes DESC
    LIMIT 1
"""))

## 6. Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos

In [0]:
display(spark.sql("""
    WITH max_ano AS (
        SELECT MAX(ano_lancamento) as ano_max
        FROM gold.dim_movies
        WHERE status_filme = 'Lançado' AND data_lancamento <= current_date()
    )
    SELECT c.nome_empresa, SUM(f.lucro_usd) AS lucro_total_usd
    FROM gold.dim_companies c
    JOIN gold.bridge_movie_company b ON c.sk_company_id = b.sk_company_id
    JOIN gold.dim_movies m ON b.sk_movie_id = m.sk_movie_id
    JOIN gold.fact_movies_performance f ON m.sk_movie_id = f.sk_movie_id
    CROSS JOIN max_ano
    WHERE m.ano_lancamento >= (max_ano.ano_max - 5)
    GROUP BY c.nome_empresa
    ORDER BY lucro_total_usd DESC
    LIMIT 1
"""))